# Beta Mapping Pipeline 
- BrainSense Surveys
- large data cohort (NeuroCure > rawdata_v3)
- one session per patient (Fu12m preferably, otherwise preferably longterm FU, otherwise Fu3m)

In [3]:
import os
import sys
import importlib
from importlib import reload 
from dataclasses import dataclass, field, fields
from itertools import compress
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.io as sio
from scipy import signal
from scipy.signal import spectrogram, butter, filtfilt, freqz
try:
    from scipy.signal import hann
except:
    from scipy.signal.windows import hann
from scipy.integrate import simps
from scipy import stats

import pickle
import json
import csv

#mne
import mne_bids
import mne
from mne.time_frequency import tfr_morlet 
from mne.stats import permutation_cluster_test
from mne.preprocessing import ICA, create_ecg_epochs


In [4]:
# from PerceiveImport.classes import (
#     main_class, modality_class, metadata_class,
#     session_class, condition_class, task_class,
#     contact_class, run_class
# )

# import PerceiveImport.methods.load_rawfile as load_rawfile
# import PerceiveImport.methods.find_folders as PyPerceive_find_folders
# import PerceiveImport.methods.metadata_helpers as metaHelpers

# utils
import beta_profile.utils.find_folders as find_folders
import beta_profile.utils.io as io
import beta_profile.utils.sub_session_dict as sub_session_dict

# tfr
import beta_profile.beta_profile.tfr_preprocessing as tfr
import beta_profile.beta_profile.tfr_plots as tfr_plots

# plots
import beta_profile.beta_profile.power_spectra_plots as power_spectra
import beta_profile.beta_profile.time_series_ecg_artifact as time_series

# beta profile
import beta_profile.beta_profile.calculate_features as calculate_features
import beta_profile.beta_profile.beta_mapping_pipeline as beta_mapping

importlib.reload(find_folders)
importlib.reload(io)
importlib.reload(tfr)
importlib.reload(power_spectra)
importlib.reload(calculate_features)
importlib.reload(time_series)
importlib.reload(sub_session_dict)
importlib.reload(tfr_plots)
importlib.reload(beta_mapping)

<module 'beta_profile.beta_profile.beta_mapping_pipeline' from '/Users/jenniferbehnke/code/Beta_profile_project/Beta_profile/src/beta_profile/beta_profile/beta_mapping_pipeline.py'>

In [ ]:
# load if you want to see complete Dataframes
pd.set_option("display.max_rows", None)

In [1]:
# for interactive plots
#%matplotlib notebook

# for static plots
%matplotlib inline 

## Data overview

In [74]:
sub_session_dict = io.find_sessions_per_sub(condition="MedOff", modality="survey")

In [75]:
sub_session_dict = sub_session_dict[1]
sub_session_dict

{'sub-007': ['Fu14m'],
 'sub-011': ['Fu37m', 'Fu50m'],
 'sub-013': ['Fu31m'],
 'sub-014': ['Fu12m'],
 'sub-015': ['Fu05m'],
 'sub-016': ['Fu25m'],
 'sub-017': ['Fu11m', 'Fu03m'],
 'sub-019': ['Fu12m', 'Fu02m', 'Fu23m'],
 'sub-020': ['Fu22m', 'Fu13m'],
 'sub-021': ['Fu12m', 'Fu36m', 'Fu03m'],
 'sub-022': ['Fu00m'],
 'sub-023': ['Fu13m'],
 'sub-024': ['Fu00m', 'Fu26m', 'Fu39m', 'Fu11m', 'Fu03m', 'Fu17m'],
 'sub-025': ['Fu00m', 'Fu12m', 'Fu04m'],
 'sub-026': ['Fu12m', 'Fu03m'],
 'sub-028': ['Fu00m', 'Fu12m', 'Fu24m'],
 'sub-029': ['Fu00m', 'Fu36m', 'Fu16m', 'Fu25m', 'Fu03m'],
 'sub-030': ['Fu00m', 'Fu12m', 'Fu35m', 'Fu03m'],
 'sub-031': ['Fu00m', 'Fu03m'],
 'sub-032': ['Fu00m', 'Fu03m'],
 'sub-033': ['Fu12m', 'Fu24m', 'Fu03m', 'Fu17m'],
 'sub-036': ['Fu12m', 'Fu24m', 'Fu18m'],
 'sub-038': ['Fu00m', 'Fu03m'],
 'sub-039': ['Fu13m'],
 'sub-040': ['Fu12m', 'Fu24m', 'Fu03m'],
 'sub-041': ['Fu17m'],
 'sub-042': ['Fu13m'],
 'sub-043': ['Fu13m'],
 'sub-044': ['Fu13m'],
 'sub-045': ['Fu12m', 'Fu03

In [8]:
sub_list = sub_session_dict.get_all_included_subjects()
len(sub_list)

68

## Main function to plot, clean and save data (per sub, hem, channel group):
- unfiltered, uncleaned time series, Power Spectra (band-pass filtered 5-95 Hz), Time Frequency
- plot uncleaned automatically into "raw" folder
- if ECG cleaning performed -> save clean data into "clean" folder
- if data already clean -> save original data into "clean" folder


In [77]:
hemispheres = ["Left", "Right"]
# 056 Right not cleaned
# 077 Left not sure what that low frequency activity is
# 083 left not cleaned
# 088 Left still noisy after cleaning
# 108 Key Error "RingL"

for hem in hemispheres: 

    plot_uncleaned = beta_mapping.plot_and_clean_raw_data(
        sub = "114",
        hemisphere=hem,
        session = "Fu03m",
        condition="m0s0",
    )